# UNBLIND_01 — DLA-tier CDDF unblinding figures  (Team 1, notebook B)

## 🔴 READ ME FIRST — this notebook plots **UNBLINDED real-LOA (DESI DR2 Loa main-dark) results**

* Every rendered figure and printed cell below contains **private real-LOA result values**
  (dN/dX, Ω_HI, f(N), per-z tables). These are **embargoed** until PI unblinding sign-off.
* **The committed copy of this notebook MUST have all outputs cleared**
  (`jupyter nbconvert --clear-output --inplace notebooks/UNBLIND_01_dla_cddf.ipynb`;
  Team1-A's `CDDF_analysis.unblind.assert_no_outputs` is the programmatic check).
  A committed executed cell = a privacy leak.
* **Saved figures belong in the private notes repo** (`~/desi_gpy_dla_notes/`), never in the code repo.
  This notebook writes PNGs only when the env var `UNBLIND_FIG_DIR` points at a scratch dir.
* No real-LOA number is hard-coded anywhere here: every plotted value is pulled from the
  guarded `HeadlineData` **at render time**, behind Team1-A's provenance guard.

**Provenance chain (headline):** `run_loa0_headline_full.py` (a *config-only* FP-model override
of committed job `52266001`) → stamped JSON `track_c_tf_loa_loa0_restamped.json`
(`code_commit=d496f42`, guard status **RE_DERIVABLE**). The un-restamped sibling carries
`code_commit="unknown"` and is used below only as a **negative test** the guard must reject.

## 0 · Provenance guard, schema validation, and data loading  (Team1-A foundation)

The guard runs **before any measurement array is read**. We use Team1-A's committed package
`CDDF_analysis.unblind`:

* `check_artifact(path, routine)` — classifies provenance and **raises** unless `RE_DERIVABLE`
  (rejects `NOT_STAMPED` / `DIRTY` / `ORPHANED` / `NOT_ANCESTOR`).
* `load_headline(path, routine)` — runs that guard **and** the schema validator, then returns a
  tidy `HeadlineData` (numpy arrays in memory only) with three **distinct** per-z regime flags.

If the package is not importable we fall back to a clearly-marked **temporary stub** so the two
notebooks still integrate.

In [ ]:
import os, sys, json, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
# %matplotlib inline MUST come after the imports above
%matplotlib inline

# repo root on sys.path so CDDF_analysis.unblind (Team1-A) is importable
_HERE = os.getcwd()
_REPO = _HERE if os.path.basename(_HERE) != "notebooks" else os.path.dirname(_HERE)
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)

# ---- PARAMETERS (paths + expected provenance; NO measurement literals) -------------------
ART = dict(
    loa0_headline=dict(
        path="/scratch/cavestru_root/cavestru0/mfho/cddf_o3_realdata/track_c/tf_loa/"
             "track_c_tf_loa_loa0_restamped.json",
        routine="CDDF_analysis/diagnostics/bal_metal_fp/arbiter/run_loa0_headline_full.py",
        expected_commit="d496f42", expect_fp="loa0", label="loa0 (headline)"),
    archival_pm=dict(
        path="/scratch/cavestru_root/cavestru0/mfho/cddf_o3_realdata/track_c/tf_loa/"
             "track_c_tf_loa.json",
        routine="CDDF_analysis/hbi/track_c_tf_loa.py",
        expected_commit="f1784fc", expect_fp=None, label="purity_mixture (archival)"),
    # NEGATIVE TEST ONLY — code_commit="unknown"; the guard MUST reject this. Never plotted.
    unstamped_negative=dict(
        path="/scratch/cavestru_root/cavestru0/mfho/cddf_o3_realdata/track_c/tf_loa/"
             "track_c_tf_loa_loa0.json",
        routine="CDDF_analysis/diagnostics/bal_metal_fp/arbiter/run_loa0_headline_full.py",
        label="UNSTAMPED (negative test)"),
)

# save figures only to a scratch dir if the env var is set (never into the repo)
FIG_DIR = os.environ.get("UNBLIND_FIG_DIR")
def _savefig(fig, name):
    if FIG_DIR:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name), dpi=110, bbox_inches="tight")

In [ ]:
import os, sys
_REPO = os.path.abspath(os.environ.get("UNBLIND_REPO_ROOT", os.getcwd()))
while _REPO != "/" and not os.path.isdir(os.path.join(_REPO, ".git")):
    _REPO = os.path.dirname(_REPO)
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)

# ---------------------------------------------------------------------------
# PROVENANCE GUARD — must FAIL CLOSED.
#
# There was a `# TEMPORARY STUB` fallback here. It was removed on 2026-07-09 after an
# adversarial review showed it fails OPEN: the stub did no git inspection at all (a bare
# `^[0-9a-f]{7,40}$` regex), so it accepted a fabricated SHA and the ORPHANED london-0
# stamp `cff73cb` as RE_DERIVABLE. The only signal was a print, which disappears the moment
# outputs are cleared for commit. An unblinding notebook's central safety mechanism must
# raise when the real guard cannot be imported -- never substitute a weaker one.
# ---------------------------------------------------------------------------
from CDDF_analysis.unblind import check_artifact, load_headline, carried_systematics  # noqa: E402


In [ ]:
# ---- NEGATIVE TEST: the guard must REJECT the un-stamped (code_commit="unknown") artifact ----
# Proves the guard fires BEFORE any measurement value is touched.
neg = ART["unstamped_negative"]
_rejected = False
try:
    check_artifact(neg["path"], neg["routine"])
except ProvenanceError as e:
    _rejected = True
    print("GUARD REJECTED the unstamped artifact (as required):\n   ", str(e))
assert _rejected, "SECURITY FAILURE: guard accepted an artifact with code_commit='unknown'."
print("\nNegative test PASSED — an un-provenanced artifact cannot be unblinded.")

In [ ]:
# ---- load the guarded artifacts (guard + schema run INSIDE load_headline, before arrays used) ----
H = ART["loa0_headline"];  A = ART["archival_pm"]
loa0 = load_headline(H["path"], H["routine"])
print(f"GUARD PASSED  {H['label']:22s}  status={loa0.provenance.status}  "
      f"fp={loa0.fp_estimator}  commit={(loa0.code_commit or '')[:12]}")
assert loa0.fp_estimator in (H["expect_fp"], None) or not _USING_TEAM1A
try:
    arch = load_headline(A["path"], A["routine"])
    print(f"GUARD PASSED  {A['label']:22s}  status={arch.provenance.status}  "
          f"fp={arch.fp_estimator}  commit={(arch.code_commit or '')[:12]}")
except ProvenanceError as e:
    arch = None
    print(f"[warn] archival rejected by guard ({e}); FP-bracket panel shows headline only.")

# quick structural echo (shapes/flags only — NO result values)
lc = loa0.logN_centers
print(f"\nheadline: {len(loa0.zbins)-1} z-bins; logN grid {lc.min():.2f}..{lc.max():.2f} ({lc.size} pts)")

## 0b · z-bin support masks — **the three extrapolations do not coincide**

`load_headline` surfaces three **distinct** per-z regime flags on `HeadlineData.zflags`
(`CDDF_analysis/unblind/loader.py:ZBinFlags`) so no partially-supported bin is ever drawn like a
fully-validated one:

| mask | definition | meaning |
|---|---|---|
| `beyond_calibration` | `metadata.z_extrapolated` | g(N,z) has no mock-truth support (excluded from headline) |
| `beyond_v2_fit` | bin **lower edge ≥ `metadata.v2_z_fit_hi`** | mean-flux / τ_eff model is extrapolated past its fit ceiling |
| `partial_truth_support` | bin **straddles `metadata.max_truth_z`** | truth support over only part of the bin |
| `perz_fN_empty` | **`perz_fN.f` all non-finite** (derived from the array) | the per-z CDDF panel *does not exist* here (fine z-fit grid ends at `v2_z_fit_hi`) — **not** "no absorbers" |

**The trap this exposes** (and which the loader prints a loud WARNING about): the **z = 3.75 bin
([3.5, 4.0])** lies *entirely above the v2 fit ceiling* (3.5) **and** straddles the truth cap
(`max_truth_z ≈ 3.79`), yet it is **not** in `z_extrapolated`. `zflags.v2_beyond_but_calibrated`
names exactly this index. A **second, independent trap**: that same z = 3.75 bin has
`extrapolated = False` *and* an entirely empty `perz_fN.f`, so `z_extrapolated` tells you neither that
the bin is above the model ceiling nor that its CDDF panel is missing — hence the derived `perz_fN_empty`
flag. Every figure below encodes these masks with distinct styles + a legend.

_Falsifier:_ if `beyond_v2_fit` picked out **no** bin that `z_extrapolated` misses
(`v2_beyond_but_calibrated == ()`), the concern would be moot. It is not (see printout).

In [ ]:
zf = loa0.zflags
z_lo, z_hi, z_ctr = zf.zbin_lo, zf.zbin_hi, zf.z_centers
v2_hi, max_tz = zf.v2_z_fit_hi, zf.max_truth_z
# fourth DERIVED flag: the per-z CDDF f(N|z) array is entirely non-finite in this bin.
# Derived from the ACTUAL fN arrays (not assumed from a z threshold): the fine z-fit grid
# ends at v2_z_fit_hi=3.5 (cddf_catalog_hbi.py:2500), so perz_fN has NO support above it —
# distinct from "no absorbers". These bins still have GENUINE binned dN/dX / Omega (figs 2-3).
_perz_fN_empty = np.array(
    [np.isfinite(loa0.fN["f"][i]).sum() == 0 for i in range(len(z_ctr))], bool)
MASK = dict(
    beyond_calibration    = zf.beyond_calibration,
    beyond_v2_fit         = zf.beyond_v2_fit,
    partial_truth_support = zf.partial_truth_support,
    perz_fN_empty         = _perz_fN_empty,
    z_gt_4                = zf.zbin_lo >= 4.0,
)
def bin_flags(i):
    t = []
    if MASK["beyond_calibration"][i]:    t.append("beyond_calibration")
    if MASK["beyond_v2_fit"][i]:         t.append("beyond_v2_fit")
    if MASK["partial_truth_support"][i]: t.append("partial_truth")
    if MASK["perz_fN_empty"][i]:         t.append("perz_fN_empty")
    return t
# 'supported' for MARKER styling ignores perz_fN_empty (dN/dX & Omega still measured there);
# perz_fN_empty governs only whether the f(N|z) PANEL can be drawn.
def bin_supported(i):
    return not (MASK["beyond_calibration"][i] or MASK["beyond_v2_fit"][i]
                or MASK["partial_truth_support"][i])

_SEV = dict(ok="#1f4e79", partial="#b8860b", v2="#d2691e", calib="#c1121f")
def bin_style(i):
    if MASK["beyond_calibration"][i]:
        return dict(color=_SEV["calib"], marker="o", mfc="white", ls="--", hatch="xxx")
    if MASK["beyond_v2_fit"][i]:
        return dict(color=_SEV["v2"], marker="s", mfc="white", ls="--", hatch="//")
    if MASK["partial_truth_support"][i]:
        return dict(color=_SEV["partial"], marker="D", mfc="#ffe9a8", ls=":", hatch="..")
    return dict(color=_SEV["ok"], marker="o", mfc=_SEV["ok"], ls="-", hatch=None)

# ---- shared accessors on HeadlineData (NO result values printed) ----
def perz_q(hd, obs, lim):
    d = hd.perz[lim][obs]
    return d["MAP"], d["q16"], d["q84"], d["q025"], d["q975"]
def integ_q(hd, obs, lim, scale=1.0):
    I = hd.integrated[lim][obs]
    return {k: (I[k]*scale if np.isfinite(I[k]) else np.nan)
            for k in ("MAP","q16","q84","q025","q975","std")}
FLOOR_N = min(float(l) for l in loa0.limits)   # headline N_HI floor (== perz_fN floor, 20.0)

print("z-bin support masks (True = flagged):")
for i in range(len(z_ctr)):
    print(f"  z={z_ctr[i]:.3f}  edges[{z_lo[i]:.2f},{z_hi[i]:.2f}]  "
          f"flags={bin_flags(i) or ['fully-supported']}")
print(f"\nv2_z_fit_hi={v2_hi}  max_truth_z={max_tz:.4f}  "
      f"v2_beyond_but_calibrated={getattr(zf,'v2_beyond_but_calibrated',None)}")
assert MASK["beyond_v2_fit"].sum() > MASK["beyond_calibration"].sum(), \
    "expected beyond_v2_fit to flag MORE bins than z_extrapolated (the z=3.75 trap)."
print("CHECK: beyond_v2_fit is a strict superset of z_extrapolated — the z=3.75 trap is caught.")
# The per-z CDDF panels exist ONLY where perz_fN is populated; dN/dX & Omega are measured in ALL bins.
_n_fN = int((~MASK["perz_fN_empty"]).sum()); _n_meas = int(np.isfinite(perz_q(loa0,"dndx","20.3")[0]).sum())
print(f"per-z CDDF panels populatable: {_n_fN}/{len(z_ctr)}  |  "
      f"binned dN/dX(>=20.3) bins with finite MAP: {_n_meas}/{len(z_ctr)}")
print("perz_fN_empty bins (NO f(N|z) panel; still have genuine dN/dX & Omega):",
      [f'z={z_ctr[i]:.3f}' for i in range(len(z_ctr)) if MASK['perz_fN_empty'][i]])

## 1 · f(N, z) — column-density distribution per redshift bin (per-z panel grid)

**What this shows:** one panel per z-bin: the MAP differential f(N | z) (points) with its 68% and 95%
MC bands vs log N_HI. The floor (headline N_HI limit) is marked. Each panel's title/annotation names which
support masks apply; flagged bins use open/coloured markers + hatched bands and are **never** styled like a
fully-supported bin.

🔴 **Empty per-z CDDF panels are NOT null measurements.** `perz_fN.f` is populated only for bins below the
fine z-fit ceiling (`v2_z_fit_hi = 3.5`); it is **entirely non-finite for z = 3.75 and z = 4.125**. Those
panels are explicitly detected (`perz_fN_empty`), shaded, and labelled "NO PER-z CDDF AT THIS REDSHIFT —
fine z-fit grid ends at v2_z_fit_hi" — a blank axis that reads as "no absorbers" is the failure mode we avoid.
The **binned dN/dX and Ω_HI for those same bins are genuine and are plotted in figs 2–3.**

**What would falsify the interpretation:** (a) a flagged panel that tracks the low-z measured panels with a
comparably tight band would undercut "extrapolation is uncertain"; (b) a measured panel whose 95% band does
**not** contain its 68% band would indicate a broken band; (c) f(N) rising (not falling) toward high N_HI
would contradict a physical CDDF.

In [ ]:
fN = loa0.fN
nz = len(z_ctr)
fig, axes = plt.subplots(1, nz, figsize=(3.1*nz, 3.6), sharey=True)
if nz == 1: axes = [axes]
for i in range(nz):
    ax = axes[i]; x = lc
    f = fN["f"][i]
    lo68, hi68 = fN["f68_lo"][i], fN["f68_hi"][i]
    lo95, hi95 = fN["f95_lo"][i], fN["f95_hi"][i]
    st = bin_style(i)
    finite = np.isfinite(f) & (f > 0)
    if finite.sum() == 0:
        # NOT "no absorbers": the fine z-fit grid ends at v2_z_fit_hi -> perz_fN has no
        # support here. Shade the panel so it can never be misread as a measured small f(N).
        ax.set_facecolor("#f2f2f2")
        ax.text(0.5, 0.55, "NO PER-z CDDF AT THIS REDSHIFT", ha="center", va="center",
                transform=ax.transAxes, fontsize=8.5, color=_SEV["calib"], weight="bold")
        ax.text(0.5, 0.40, f"fine z-fit grid ends at\nv2_z_fit_hi = {v2_hi:g}\n"
                "(dN/dX & Ω still measured — see figs 2–3)", ha="center", va="center",
                transform=ax.transAxes, fontsize=7.5, color="0.25")
    else:
        m95 = np.isfinite(lo95) & np.isfinite(hi95) & (lo95 > 0)
        m68 = np.isfinite(lo68) & np.isfinite(hi68) & (lo68 > 0)
        ax.fill_between(x[m95], lo95[m95], hi95[m95], color=st["color"], alpha=0.12, hatch=st["hatch"])
        ax.fill_between(x[m68], lo68[m68], hi68[m68], color=st["color"], alpha=0.28, hatch=st["hatch"])
        ax.plot(x[finite], f[finite], st["marker"], ms=3.5, mfc=st["mfc"], mec=st["color"],
                color=st["color"])
    ax.axvline(FLOOR_N, color="0.4", lw=0.8, ls=":")
    ax.set_yscale("log"); ax.set_xlim(19.3, 22.5); ax.set_xlabel(r"$\log_{10} N_{\rm HI}$")
    ax.set_title(f"z = {z_ctr[i]:.2f}  [{z_lo[i]:.1f},{z_hi[i]:.1f}]", fontsize=10, color=st["color"])
    tags = bin_flags(i)
    if tags:
        ax.text(0.03, 0.03, "\n".join("⚠ "+t for t in tags), transform=ax.transAxes,
                fontsize=7.5, color=st["color"], va="bottom",
                bbox=dict(fc="white", ec=st["color"], lw=0.8, alpha=0.85))
    if MASK["z_gt_4"][i]:
        ax.text(0.5, 0.92, "z>4: NO mock validates", transform=ax.transAxes, ha="center",
                fontsize=8, color=_SEV["calib"], weight="bold")
axes[0].set_ylabel(r"$f(N_{\rm HI}\,|\,z)$")
handles = [
    Line2D([],[], color=_SEV["ok"], marker="o", ls="-", label="fully supported"),
    Line2D([],[], color=_SEV["partial"], marker="D", mfc="#ffe9a8", ls=":", label="partial truth support"),
    Line2D([],[], color=_SEV["v2"], marker="s", mfc="white", ls="--", label="beyond v2 τ-eff fit"),
    Line2D([],[], color=_SEV["calib"], marker="o", mfc="white", ls="--", label="beyond calibration (z_extrap)"),
]
fig.legend(handles=handles, loc="upper center", ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("f(N | z) per z-bin  (dotted line = headline N_HI floor)", y=1.0, fontsize=11)
fig.tight_layout(); _savefig(fig, "fig1_fNz_panels.png"); plt.show()

## 1c · 🔴 z > 4 extrapolation — validated by **no mock**

The 2LPT-0 mock truth **caps at z ≈ 3.5–3.79** (`max_truth_z ≈ 3.79`). Any point with z > 4 (the
**z = 4.125** bin, `[4.0, 4.25]`) is therefore recovered with **no mock available to bound its bias** — an
**unquantified / unbounded extrapolation systematic**, not a measurement. Team1-A's systematics table lists
it as `size="UNBOUNDED", status=UNVERIFIED`. It is flagged `beyond_calibration` (and `beyond_v2_fit`), and
everywhere it appears below (dN/dX(z), Ω(z)) it is drawn open + dashed + hatched and labelled.

🔴 **State the asymmetry plainly.** The two families of high-z quantities are NOT on the same footing:
* **dN/dX(z) and Ω_HI(z) *exist* at z > 4** (detections + path length + truth above 3.5 fold into the
  high-z report bins; `track_c_tf_loa.py:925-927`), but their **completeness calibration is extrapolated**
  (no mock truth above the cap) — so the point exists, its bias is unbounded.
* **The per-z CDDF f(N | z) does *not* exist at z > 4 at all** — nor even at z = 3.75 — because the fine
  z-fit grid ends at `v2_z_fit_hi = 3.5`. A missing f(N | z) panel there is an *absence of the estimator*,
  **not** a measurement of a small f. Do not let a reader infer otherwise.

**Old-vs-new (`calc_cddf` job 52266000 vs HBI job 52266001):** a `calc_cddf` results JSON for job
52266000 was **not found on scratch** — only `regen_logs/regen_old_loa_calccddf_52266000.{log,err}` exist,
with no serialized CDDF. The old-vs-new z > 4 comparison is therefore **omitted** (not fabricated).

In [ ]:
import glob
_hits = [p for p in glob.glob(
    "/scratch/cavestru_root/cavestru0/mfho/**/*calc_cddf*.json", recursive=True)
    if "52266000" in p or "loa" in p.lower()]
print("calc_cddf JSON candidates for old-vs-new z>4 comparison:", _hits or "NONE FOUND")
if not _hits:
    print("-> old-vs-new z>4 comparison OMITTED (no archival calc_cddf JSON to plot; not fabricated).")

## 2 · dN/dX(z) — DLA line density at both N_HI limits

**What this shows:** the incidence rate dN/dX per z-bin at N_HI ≥ 20.0 and ≥ 20.3, with 68% (thick) and
95% (thin) MC error bars. Flagged bins use the §0b mask styles; the z > 4 bin gets an open marker, dashed
connector and a "no mock" annotation.

**What would falsify the interpretation:** dN/dX(≥20.0) falling **below** dN/dX(≥20.3) in any bin (the
≥20.0 sample strictly contains the ≥20.3 sample, so it must be ≥); or a trend that tracks the calibration
boundary rather than physical evolution.

In [ ]:
def plot_perz(ax, hd, obs, scale=1.0, limits=("20.0","20.3"), colors=("#1f77b4","#d62728")):
    for lim, base in zip(limits, colors):
        MAP, q16, q84, q025, q975 = (a*scale for a in perz_q(hd, obs, lim))
        for i in range(len(z_ctr)):
            if not np.isfinite(MAP[i]): continue
            st = bin_style(i); flagged = not bin_supported(i)
            col = st["color"] if flagged else base
            if np.isfinite(q025[i]) and np.isfinite(q975[i]):
                ax.plot([z_ctr[i]]*2, [q025[i], q975[i]], color=col, lw=0.9, alpha=0.7, zorder=2)
            if np.isfinite(q16[i]) and np.isfinite(q84[i]):
                ax.plot([z_ctr[i]]*2, [q16[i], q84[i]], color=col, lw=2.6, alpha=0.9, zorder=3)
            else:
                ax.annotate("no band", (z_ctr[i], MAP[i]), fontsize=6.5, color=col,
                            xytext=(3,3), textcoords="offset points")
            ax.plot(z_ctr[i], MAP[i], st["marker"], ms=7 if flagged else 6,
                    mfc="white" if flagged else base, mec=col, zorder=4)
        sup = np.array([bin_supported(i) and np.isfinite(MAP[i]) for i in range(len(z_ctr))])
        ax.plot(z_ctr[sup], MAP[sup], "-", color=base, lw=1.3, alpha=0.8, label=f"N_HI ≥ {lim}")
        if (~sup).any():
            fin = np.isfinite(MAP)
            ax.plot(z_ctr[fin], MAP[fin], "--", color=base, lw=0.8, alpha=0.4)
    for i in np.where(MASK["z_gt_4"])[0]:
        ax.axvspan(z_lo[i], z_hi[i], color=_SEV["calib"], alpha=0.06)
        ax.text(z_ctr[i], ax.get_ylim()[1], "z>4\nno mock", ha="center", va="top",
                fontsize=7.5, color=_SEV["calib"], weight="bold")

fig, ax = plt.subplots(figsize=(6.6, 4.4))
plot_perz(ax, loa0, "dndx")
ax.set_xlabel("absorber redshift z"); ax.set_ylabel("dN/dX")
ax.set_title("DLA line density dN/dX(z)  (loa0 headline; open+dashed = flagged bins)")
ax.legend(loc="best", fontsize=9)
fig.tight_layout(); _savefig(fig, "fig2_dndx_z.png"); plt.show()

## 3 · Ω_HI(z) — neutral-gas mass density at both limits (×10³)

**What this shows:** 10³ · Ω_HI per z-bin at N_HI ≥ 20.0 and ≥ 20.3 with MC error bars. Bins whose MC band
is non-finite (data-thin high-z slices) are drawn as a MAP marker with a "no band" note — never a fake band.
Flagged bins and the z > 4 bin follow the §0b styles.

**What would falsify the interpretation:** Ω(≥20.0) below Ω(≥20.3) (nesting violation); or the z > 4 point
carrying a band as tight as the calibrated bins (which would misrepresent an unvalidated extrapolation).

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.4))
plot_perz(ax, loa0, "omega", scale=1e3)
ax.set_xlabel("absorber redshift z"); ax.set_ylabel(r"$10^{3}\,\Omega_{\rm HI}$")
ax.set_title(r"$\Omega_{\rm HI}(z)$  (loa0 headline; open+dashed = flagged bins)")
ax.legend(loc="best", fontsize=9)
fig.tight_layout(); _savefig(fig, "fig3_omega_z.png"); plt.show()

## 4 · Integrated scalars — loa0 headline vs purity_mixture archival = **FP-model bracket**

**What this shows:** the integrated dN/dX and Ω_HI at N_HI ≥ 20.0 and ≥ 20.3, comparing the **loa0** FP model
(the headline) against the **purity_mixture** archival run side by side. The project reports **both** as an
FP-model bracket (loa0 = headline). Two uncertainty bands are overlaid and kept visually distinct: the
**statistical MC band** (from the artifact's integrated quantiles) as the inner error bar, and the **carried
systematic band** as an outer whisker — the latter is **one-sided and obs/limit-specific**, not a symmetric ±.

🔴 **The carried systematic is asymmetric.** The dominant **deep-tail / mean-flux transfer** term is *one-sided
and downward on Ω*: transferring the 2LPT-0-frozen calibration to a different forest recipe *under*-recovers Ω
(london-0 mock R0 < 1), so the correction — if applied — goes **up** by 1/R0 (≈ +14.7% at ≥20.3, +13.6% at
≥20.0). It is far smaller on dN/dX (≈ +1.9% at ≥20.3). The **BAL-FP** residual is one-sided the *other* way
(over-count → correcting lowers the value ~2–6%). So the outer whisker runs from `MAP·(1−BAL)` up to `MAP/R0`.

**What would falsify the interpretation:** if the loa0 vs purity_mixture separation exceeded the carried
systematic interval, the two FP models would not "bracket"; if the MC band alone spanned the FP-model gap, the
FP choice would be statistically irrelevant.

In [ ]:
# ---- CARRIED SYSTEMATICS: asymmetric, one-sided, obs/limit-specific --------------------
# BAL-FP residual: Team1-A table (VERIFIED); one-sided OVER-count -> correcting lowers Omega.
# deep-tail / mean-flux transfer: RE-CLASSIFIED here per the 2026-07-09 cross-check as
#   ORPHANED (not VERIFIED). Its magnitude is (a) a committed PROSE literal "~12% downward"
#   in track_c_tf_loa.py:831-834 (a report string, not computed), and (b) a london-0 MOCK
#   transfer artifact stamped cff73cb which is an ancestor of HEAD but does NOT contain its
#   routine track_c_tf_london0.py -> not re-derivable from committed code.
# It is ONE-SIDED & DOWNWARD on Omega: transferring the 2LPT-0-frozen calibration to a
# different forest recipe UNDER-recovers Omega (R0<1); the correction, if applied, is UP by 1/R0.
SYST_ROWS = carried_systematics()
def _syst_row(key): return next((r for r in SYST_ROWS if key in r.name), None)
def _frac(size_str):
    nums = [float(x) for x in re.findall(r"[0-9]+\.?[0-9]*", size_str)]
    return max(nums)/100.0 if nums else 0.0
R_BAL = _syst_row("BAL"); BAL_STATUS = getattr(R_BAL, "status", "UNVERIFIED")
FRAC_BAL = _frac(R_BAL.size) if R_BAL else 0.06     # ~2-6% one-sided over-count (correct DOWN)

# london-0 mock transfer recovery R0 (PUBLIC mock; READ from the artifact, not hard-coded)
LONDON0_JSON = ("/scratch/cavestru_root/cavestru0/mfho/cddf_o3_realdata/track_c/"
                "tf_london0/track_c_tf_london0.json")
with open(LONDON0_JSON) as _fh:
    _l0 = json.load(_fh)
_R0 = _l0["variants"]["A"]["integrated_R0"]          # [obs][limit]["R0"]
L0_COMMIT = _l0["metadata"].get("code_commit")
def r0(obs, lim): return float(_R0[obs][lim]["R0"])
DT_STATUS = "ORPHANED"
# one-sided carried-systematic interval on an integrated MAP (obs/limit specific):
#   deep-tail transfer -> UP correction to MAP/R0 (Omega/dN-dX measured LOW by R0)
#   BAL-FP over-count  -> DOWN correction to MAP*(1-FRAC_BAL)
def syst_interval(obs, lim, m):
    return m*(1.0 - FRAC_BAL), m/r0(obs, lim)        # (BAL-corrected low, deep-tail-corrected high)

print("london-0 transfer R0 (PUBLIC mock; ORPHANED provenance @%s):" % L0_COMMIT)
for obs in ("omega","dndx"):
    for lim in ("20.0","20.3"):
        print(f"  {obs}>= {lim}: R0={r0(obs,lim):.4f}  -> deep-tail UP-correction "
              f"+{100*(1/r0(obs,lim)-1):.1f}%  (under-recovery {100*(1-r0(obs,lim)):.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6))
for ax, (obs, scale, ylab) in zip(axes, [("dndx",1.0,"dN/dX"), ("omega",1e3,r"$10^{3}\,\Omega_{\rm HI}$")]):
    xpos = {"20.0": 0, "20.3": 1}
    runs = [("loa0", loa0, "#1a5276", -0.13)]
    if arch is not None: runs.append(("purity_mixture", arch, "#a04000", 0.13))
    for name, hd, col, dx in runs:
        for lim in ("20.0","20.3"):
            I = integ_q(hd, obs, lim, scale); x = xpos[lim] + dx
            lo, hi = syst_interval(obs, lim, I["MAP"])
            ax.plot([x, x], [lo, hi], color=col, lw=6, alpha=0.18, solid_capstyle="butt", zorder=1)
            ax.plot([x, x], [I["q025"], I["q975"]], color=col, lw=1.4, alpha=0.8, zorder=2)
            ax.plot([x, x], [I["q16"], I["q84"]], color=col, lw=3.4, alpha=0.95, zorder=3)
            ax.plot(x, I["MAP"], "o", ms=7, mfc="white", mec=col, zorder=4,
                    label=name if lim=="20.0" else None)
    ax.set_xticks([0,1]); ax.set_xticklabels([r"$N_{\rm HI}\geq20.0$", r"$N_{\rm HI}\geq20.3$"])
    ax.set_ylabel(ylab); ax.set_title(f"integrated {obs}"); ax.legend(loc="best", fontsize=9)
band_handles = [
    Line2D([],[], color="0.3", lw=3.4, label="stat MC 68% (artifact)"),
    Line2D([],[], color="0.3", lw=1.4, label="stat MC 95% (artifact)"),
    Line2D([],[], color="0.3", lw=6, alpha=0.18,
           label=f"carried syst (one-sided): deep-tail UP on Ω [ORPHANED] + BAL down ~{100*FRAC_BAL:.0f}%"),
]
fig.suptitle("Integrated dN/dX & Ω_HI: loa0 (headline) vs purity_mixture (archival) — FP-model bracket",
             y=1.16, fontsize=11)
fig.legend(handles=band_handles, loc="upper center", ncol=3, fontsize=8.0, frameon=False,
           bbox_to_anchor=(0.5, 1.06))
fig.tight_layout(); _savefig(fig, "fig4_integrated_bracket.png"); plt.show()

## 5 · Systematics band overlay — one-sided budget; **the statistical band is not the total error**

**What this shows:** for the two headline integrated scalars at N_HI ≥ 20.3, the loa0 MAP with the statistical
MC 68%/95% bands (from the artifact) plus the **two carried, OUTSIDE-the-MC systematics drawn as ONE-SIDED
arrows** in their true directions:
* **deep-tail / mean-flux transfer** — one-sided **UP** on Ω (measured LOW; correction target = `MAP/R0`,
  ≈ +14.7% at ≥20.3 from the london-0 mock R0). Only ≈ +1.9% on dN/dX — so it is overwhelmingly an **Ω**
  systematic; **no 12% band is drawn on dN/dX**.
* **BAL-FP residual** — one-sided **DOWN** (over-count; correcting lowers the value ~2–6%).

The two point in **opposite** directions, so they do not "add" into a symmetric ±; the honest picture is an
asymmetric interval. The deep-tail term is the **dominant systematic on Ω** and, per the 2026-07-09
cross-check, is **ORPHANED** (committed prose literal in `track_c_tf_loa.py:831-834` + london-0 mock artifact
stamped `cff73cb`, an ancestor of HEAD that does **not** contain its routine `track_c_tf_london0.py`) —
**not** the `VERIFIED` that Team1-A's table currently records for it. That discrepancy is flagged for the panel.

🔴 **PI DECISION NOT YET MADE — do not silently choose.** Whether the deep-tail transfer term is **applied as
a correction** (multiply Ω by 1/R0 ⇒ Ω shifts *up* ≈ 15% at ≥20.3) or **carried as a one-sided error band**
(Ω stays at MAP, with a one-sided upward uncertainty) is a **PI decision that has not been taken**. The two
give **materially different** headline Ω. This figure draws the *carry-as-band* representation and marks the
*apply-as-correction* target (`MAP/R0`) with an arrow so both are visible.

**What would falsify the interpretation:** if the london-0 transfer R0 were ≈ 1 for Ω (no under-recovery), the
deep-tail term would vanish and Ω would be statistics-limited — it is not (R0 ≈ 0.87 at ≥20.3).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8))
for ax, (obs, scale, ylab) in zip(axes, [("dndx",1.0,"dN/dX (≥20.3)"),
                                         ("omega",1e3,r"$10^{3}\,\Omega_{\rm HI}$ (≥20.3)")]):
    I = integ_q(loa0, obs, "20.3", scale); m = I["MAP"]; x = 0
    dt_hi = m / r0(obs, "20.3")          # deep-tail UP correction target
    bal_lo = m * (1 - FRAC_BAL)          # BAL-FP DOWN correction target
    # statistical MC (symmetric)
    ax.plot([x-0.18]*2, [I["q025"], I["q975"]], color="#1f77b4", lw=3.0, alpha=0.7)
    ax.plot([x-0.18]*2, [I["q16"], I["q84"]], color="#1f77b4", lw=7.0, alpha=0.95)
    # deep-tail one-sided UP (arrow from MAP to MAP/R0)
    ax.annotate("", xy=(x+0.12, dt_hi), xytext=(x+0.12, m),
                arrowprops=dict(arrowstyle="-|>", color="#c1121f", lw=2.4))
    # BAL-FP one-sided DOWN (arrow from MAP to MAP*(1-frac))
    ax.annotate("", xy=(x+0.30, bal_lo), xytext=(x+0.30, m),
                arrowprops=dict(arrowstyle="-|>", color="#ff7f0e", lw=2.4))
    ax.plot(x, m, "ko", ms=7, zorder=6)
    ax.axhline(m, color="0.7", lw=0.6, ls=":")
    ax.text(x+0.14, dt_hi, f"  deep-tail UP\n  +{100*(1/r0(obs,'20.3')-1):.1f}% ({DT_STATUS})",
            fontsize=7.2, color="#c1121f", va="center")
    ax.text(x+0.32, bal_lo, f"  BAL down\n  -{100*FRAC_BAL:.0f}% ({BAL_STATUS})",
            fontsize=7.2, color="#ff7f0e", va="center")
    ax.set_xlim(-0.6, 0.9); ax.set_xticks([]); ax.set_ylabel(ylab)
    ax.set_title(f"integrated {obs.upper()} — one-sided error budget")
handles = [
    Line2D([],[], color="#1f77b4", lw=7, label="statistical MC 68% (from artifact)"),
    Line2D([],[], color="#1f77b4", lw=3, alpha=0.7, label="statistical MC 95% (from artifact)"),
    Line2D([],[], color="#c1121f", lw=2.4, marker=">", label="deep-tail transfer: one-sided UP (Ω low) — ORPHANED"),
    Line2D([],[], color="#ff7f0e", lw=2.4, marker=">", label=f"BAL-FP: one-sided DOWN ~{100*FRAC_BAL:.0f}% ({BAL_STATUS})"),
]
fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=7.8, frameon=False, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("One-sided carried systematics (deep-tail UP on Ω dominates; BAL down; opposite signs)",
             y=1.0, fontsize=10)
fig.tight_layout(); _savefig(fig, "fig5_syst_overlay.png"); plt.show()

# print Team1-A's carried-systematics table (sizes are systematic magnitudes, not values)
print("CARRIED DLA-TIER SYSTEMATICS (Team1-A table):")
if _syst_mod is not None:
    print(_syst_mod.as_table())
else:
    for r in SYST_ROWS:
        print(f"  {r.name:34s} {r.size:12s} {r.status:11s} {r.band_relation}")
print("\n🔴 CROSS-CHECK OVERRIDE (2026-07-09): the 'deep-tail / mean-flux transfer' row above is\n"
      "   RE-CLASSIFIED VERIFIED -> ORPHANED here. Its size is a committed PROSE literal\n"
      "   (track_c_tf_loa.py:831-834), and the london-0 mock artifact (@%s) is an ancestor of\n"
      "   HEAD but does NOT contain its routine track_c_tf_london0.py. It is ONE-SIDED (Ω low),\n"
      "   obs/limit-specific (Ω ~13%%, dN/dX ~2%%), NOT a symmetric ±12-13%%. Flagged for the panel."
      % L0_COMMIT)

## 6 · Literature comparison — **web-verified citations only**

**What this shows:** our loa0 Ω_HI(z) and dN/dX(z) overlaid with published measurements **web-verified this
session**. Verified entries carry a per-value provenance (paper, arXiv, table). Anything not verifiable to a
specific tabulated value is **omitted and named** rather than drawn.

**Verification status (this session):**
* **Noterdaeme et al. 2012** (A&A 547, L1 = arXiv:1210.1213), Table 2 — Ω_g^DLA(z) in five z-bins: **VERIFIED**
  (values fetched from the article). Caveats: Noterdaeme's Ω_g^DLA carries a helium factor (μ≈1.3), and their
  Table 2 tabulates **dN/dz, not dN/dX** — so it is overlaid on Ω(z) only, not on our dN/dX(z) axis.
* **Ho, Bird & Garnett 2021** (MNRAS 507, 704 = arXiv:2103.10964) — same GP method on SDSS DR16Q; its per-z
  dN/dX(z) and Ω_DLA(z) tables live in Appendix A / the DR16Q data release and were **not machine-extractable**
  this session → **UNVERIFIED → not plotted** (named in the citation table below).
* There is **no** standalone "Bird 2023 GP-DLA catalog" paper — not cited.

**What would falsify the interpretation:** if our Ω(≥20.3) sat wholly outside Noterdaeme 2012 even after
accounting for the μ (He) factor and the dN/dz-vs-dN/dX caveat, either the measurement or the comparison would
be mis-scaled.

In [ ]:
# Convert Noterdaeme's He-inclusive Omega_gas to our Omega_HI.
# N12 state mu = 1.3 (Noterdaeme 2009 Eq. 4), so the EXACT undo is /1.3.
# The engine multiplies by X_H = 0.76 (track_c_tf_loa.py:569), implying
# mu_eff = 1.316 -- a 1.2% offset. We use the exact /1.3 here.
MU_HE = 1.3          # Noterdaeme 2009 Eq. 4: mean molecular mass of the gas
OMEGA_G_TO_HI = 1.0 / MU_HE   # = 0.7692. The engine uses X_H = 0.76 (mu_eff = 1.316),
                              # a 1.2% offset; /1.3 is the exact undo of N12's stated mu.

# LITERATURE overlay values — EXTERNAL published numbers (not from the artifact). Each entry web-verified
# this session; `verified=False` entries are NOT drawn (named only).
LITERATURE = {
    "Noterdaeme2012": dict(
        cite="Noterdaeme et al. 2012", arxiv="1210.1213", journal="A&A 547, L1",
        verified=True, source="Table 2 (Ω_g^DLA, corrected-for-systematics column; σ = quoted stat)",
        z_edges=[(2.0,2.3),(2.3,2.6),(2.6,2.9),(2.9,3.2),(3.2,3.5)],
        omega1e3_direct=[v * OMEGA_G_TO_HI for v in (0.91, 0.88, 1.19, 1.44, 1.87)],
        # 2026-07-09 FIX (physics + adversarial review): Noterdaeme+2012 tabulate Omega_g^DLA,
        # which INCLUDES helium (mu ~ 1.3). Our engine reports Omega_HI (mass m_H, no mu):
        # see cddf_catalog_hbi.py:2185 'omega_total_gas = omega_HI * 1.3 (X_H=0.76)'.
        # Plotting Omega_g on an Omega_HI axis puts the literature +30% too high (our curve
        # reads 23.1% low). It compounds MULTIPLICATIVELY with the deep-tail under-recovery
        # (london-0 Omega R0 = 0.8715): 0.8715 * 0.7692 = 0.670 -> a spurious ~33% apparent
        # deficit vs Noterdaeme, none of it physical. Convert the CENTRAL, the DIRECT series
        # AND the sigma -- all three are on the Omega_g scale.
        omega1e3_corrected=[v * OMEGA_G_TO_HI for v in (0.99, 0.87, 1.04, 1.10, 1.27)],
        omega1e3_sigma=[v * OMEGA_G_TO_HI for v in (0.05, 0.04, 0.05, 0.08, 0.13)],
        caveats="N12 tabulate Ω_g (He-inclusive, μ=1.3); DIVIDED BY 1.3 here to match our Ω_HI. Their Table 2 gives dN/dz, not dN/dX -> Ω panel only.", quantity="omega"),
    "HoBirdGarnett2021": dict(
        cite="Ho, Bird & Garnett 2021", arxiv="2103.10964", journal="MNRAS 507, 704",
        verified=False, source="Appendix A / DR16Q data release (per-z tables not machine-extracted this session)",
        quantity="dndx+omega", caveats="omitted (UNVERIFIED numeric values)"),
}
nlit = LITERATURE["Noterdaeme2012"]
nz_ctr = np.array([0.5*(a+b) for a,b in nlit["z_edges"]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
MAP, q16, q84, q025, q975 = (a*1e3 for a in perz_q(loa0, "omega", "20.3"))
for i in range(len(z_ctr)):
    if not np.isfinite(MAP[i]): continue
    st = bin_style(i); flagged = not bin_supported(i)
    if np.isfinite(q16[i]):
        ax.plot([z_ctr[i]]*2, [q16[i], q84[i]], color=st["color"], lw=2.6, alpha=0.9)
    ax.plot(z_ctr[i], MAP[i], st["marker"], ms=7, mfc="white" if flagged else st["color"],
            mec=st["color"], zorder=4)
sup = np.array([bin_supported(i) and np.isfinite(MAP[i]) for i in range(len(z_ctr))])
ax.plot(z_ctr[sup], MAP[sup], "-", color="#1a5276", lw=1.3, label="this work: loa0 (≥20.3)")
ax.errorbar(nz_ctr, nlit["omega1e3_corrected"], yerr=nlit["omega1e3_sigma"], fmt="s", ms=5,
            color="#117a65", capsize=3, label=f"{nlit['cite']} (Ω_HI = Ω_g/1.3, corrected)")
ax.plot(nz_ctr, nlit["omega1e3_direct"], "x", ms=6, color="#48c9b0", alpha=0.8,
        label=f"{nlit['cite']} (Ω_HI = Ω_g/1.3, direct)")
ax.set_xlabel("z"); ax.set_ylabel(r"$10^{3}\,\Omega_{\rm HI}$")
ax.set_title("Ω_HI(z): this work vs Noterdaeme 2012"); ax.legend(fontsize=8, loc="best")
ax.text(0.02, 0.02, "caveat: Noterdaeme Ω_g has He μ≈1.3", transform=ax.transAxes,
        fontsize=7, color="#117a65", va="bottom")
ax = axes[1]
MAP, q16, q84, q025, q975 = perz_q(loa0, "dndx", "20.3")
for i in range(len(z_ctr)):
    st = bin_style(i); flagged = not bin_supported(i)
    if np.isfinite(q16[i]):
        ax.plot([z_ctr[i]]*2, [q16[i], q84[i]], color=st["color"], lw=2.6, alpha=0.9)
    ax.plot(z_ctr[i], MAP[i], st["marker"], ms=7, mfc="white" if flagged else st["color"],
            mec=st["color"], zorder=4)
sup = np.array([bin_supported(i) for i in range(len(z_ctr))])
ax.plot(z_ctr[sup], MAP[sup], "-", color="#1a5276", lw=1.3, label="this work: loa0 (≥20.3)")
ax.set_xlabel("z"); ax.set_ylabel("dN/dX")
ax.set_title("dN/dX(z): this work (no verified lit. overlay)")
ax.text(0.5, 0.5, "No web-verified literature dN/dX(z) overlay this session\n"
        "(Noterdaeme 2012 tabulate dN/dz; HBG21 per-z in DR16Q release, UNVERIFIED)",
        transform=ax.transAxes, ha="center", va="center", fontsize=8, color="0.35",
        bbox=dict(fc="white", ec="0.6", alpha=0.8))
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); _savefig(fig, "fig6_literature.png"); plt.show()

print("CITATION TABLE (web-verified this session)")
print(f"{'cite':28s} {'arXiv':11s} {'verified':9s} {'plotted':8s} source")
for v in LITERATURE.values():
    print(f"{v['cite']:28s} {v['arxiv']:11s} {str(v['verified']):9s} {str(v['verified']):8s} {v['source']}")




## Before committing this notebook

```bash
jupyter nbconvert --clear-output --inplace notebooks/UNBLIND_01_dla_cddf.ipynb
python -c "from CDDF_analysis.unblind import assert_no_outputs; assert_no_outputs('notebooks/UNBLIND_01_dla_cddf.ipynb')"
```
Confirm zero cells carry outputs, then commit. Executed figures + tables go to the **private** notes repo
(`~/desi_gpy_dla_notes/`), never here. No real-LOA result value may leave this machine un-embargoed.